In [ ]:
#Lib Imports 
import pandas as pd
import numpy as np
import warnings

# ignorar todos los warnings
warnings.filterwarnings('ignore')


from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn_pandas import DataFrameMapper
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error


from tensorflow.keras.models import Sequential, clone_model,save_model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import Callback,EarlyStopping



In [ ]:

# cargamos el dataset principal con datos historicos y variables externas
df = pd.read_csv("../data/full_data.csv", parse_dates=["Date"], dayfirst=False)

# ordenamos por fecha
df = df.sort_values("Date").reset_index(drop=True)

# mostramos info general del dataset
print("columnas disponibles:")
print(df.columns.tolist())

print("\nprimeras filas:")
print(df.head())

print("\ntipos de datos:")
print(df.dtypes)


columnas disponibles:
['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Daily_Change', 'Volatility', 'Pct_Change', 'Volume_Change_pct', 'SMA_7', 'SMA_30', 'Rolling_volatility_30', 'BTC_Close_t-1', 'BTC_Close_t-2', 'BTC_Close_t-3', 'BTC_Close_t-7', 'google_trends', 'Oil_Open', 'Oil_High', 'Oil_Low', 'Oil_Close', 'Oil_Volume', 'Oil_Volatility', 'Oil_Daily_Change', 'Oil_Volume_Change', 'Oil_Pct_Change', 'Oil_Volume_Change_pct', 'Oil_SMA_7', 'Oil_SMA_30', 'Gold_Open', 'Gold_High', 'Gold_Low', 'Gold_Close', 'Gold_Volume', 'Gold_Volatility', 'Gold_Daily_Change', 'Gold_Volume_Change', 'Gold_Pct_Change', 'Gold_Volume_Change_pct', 'Gold_SMA_7', 'Gold_SMA_30', 'USD_Open', 'USD_High', 'USD_Low', 'USD_Close', 'USD_Volatility', 'USD_Daily_Change', 'USD_Pct_Change', 'USD_SMA_7', 'USD_SMA_30', 'TNX_Close', 'TNX_High', 'TNX_Low', 'TNX_Open', 'TNX_Volatility', 'TNX_Daily_Change', 'TNX_Pct_Change', 'TNX_SMA_7', 'TNX_SMA_30', 'Is_Halving_Date', 'Block_reward', 'fng_value', 'fng_classification', 'fng_di

In [ ]:
# generacion de targets y analisis de correlacion

# generamos los 7 targets (precio de cierre futuro)
for i in range(1, 8):
    df[f"Close_t+{i}"] = df["Close"].shift(-i)

# eliminamos las filas sin datos completos (las ultimas 7)
df = df.dropna(subset=[f"Close_t+{i}" for i in range(1, 8)]).reset_index(drop=True)

# definimos las features numericas disponibles (todas menos las categoricas o de texto)
available_features = [col for col in df.columns if df[col].dtype != "object" and col not in [f"Close_t+{i}" for i in range(1, 8)]]

# lista de targets
targets = [f"Close_t+{i}" for i in range(1, 8)]


In [ ]:
# seleccion de features mas correlacionadas

selected_features = ['Date','Volume','Pct_Change','Volume_Change_pct','Volatility','SMA_7','SMA_30',
             'fng_value','fng_SMA_7','fng_SMA_30','BTC_Close_t-1','BTC_Close_t-2','BTC_Close_t-3','BTC_Close_t-7']


# dejamos solo las columnas seleccionadas y los 7 targets
keep_cols = selected_features + [f"Close_t+{i}" for i in range(1, 8)]
df = df[keep_cols].copy()

print(f"dataset final listo para entrenamiento, con {len(selected_features)} features y 7 targets\n")
print(df.head())


dataset final listo para entrenamiento, con 24 features y 7 targets

        Date      Volume  Pct_Change  Volume_Change_pct  Volatility  \
0 2018-06-01  4921460224    0.006309          -0.040114  197.390137   
1 2018-06-02  4939299840    0.013525           0.003625  198.570312   
2 2018-06-03  4851760128    0.010048          -0.017723  141.850098   
3 2018-06-04  4993169920   -0.026655           0.029146  279.779785   
4 2018-06-05  4961739776    0.015875          -0.006295  246.229980   

         SMA_7       SMA_30   Gold_Close  Gold_Volatility  Gold_Pct_Change  \
0  7396.402902  8372.248340  1294.800049        10.599976        -0.004077   
1  7437.484375  8302.234668  1294.800049        10.599976        -0.004077   
2  7487.774344  8236.217676  1294.800049        10.599976        -0.004077   
3  7541.842913  8158.095003  1293.099976         7.099976        -0.001313   
4  7564.867188  8090.727002  1297.500000         9.900024         0.003403   

   ...  BTC_Close_t-2  BTC_Close_t-

In [10]:
# features: todas las columnas numericas excepto los targets y la fecha
targets = [f"Close_t+{i}" for i in range(1, 8)]
features = [c for c in df.columns if c not in targets + ["Date"]]

# diccionarios para guardar X e Y por horizonte
X_dict = {}
Y_dict = {}

for i in range(1, 8):
    tcol = f"Close_t+{i}"
    X_dict[i] = df[features].copy()
    Y_dict[i] = df[[tcol]].copy()

# mostrar shapes para confirmar
print("resumen de shapes por horizonte (i -> X.shape -> y.shape):\n")
for i in range(1, 8):
    print(f"t+{i}: X {X_dict[i].shape} -> Y {Y_dict[i].shape}")

# ejemplo: mostrar las primeras filas del horizonte 1
print("\nprimeras filas - ejemplo horizonte t+1 (X, Y):")
print(X_dict[1].head())
print(Y_dict[1].head())


resumen de shapes por horizonte (i -> X.shape -> y.shape):

t+1: X (2684, 23) -> Y (2684, 1)
t+2: X (2684, 23) -> Y (2684, 1)
t+3: X (2684, 23) -> Y (2684, 1)
t+4: X (2684, 23) -> Y (2684, 1)
t+5: X (2684, 23) -> Y (2684, 1)
t+6: X (2684, 23) -> Y (2684, 1)
t+7: X (2684, 23) -> Y (2684, 1)

primeras filas - ejemplo horizonte t+1 (X, Y):
       Volume  Pct_Change  Volume_Change_pct  Volatility        SMA_7  \
0  4921460224    0.006309          -0.040114  197.390137  7396.402902   
1  4939299840    0.013525           0.003625  198.570312  7437.484375   
2  4851760128    0.010048          -0.017723  141.850098  7487.774344   
3  4993169920   -0.026655           0.029146  279.779785  7541.842913   
4  4961739776    0.015875          -0.006295  246.229980  7564.867188   

        SMA_30   Gold_Close  Gold_Volatility  Gold_Pct_Change   Gold_SMA_7  \
0  8372.248340  1294.800049        10.599976        -0.004077  1298.771432   
1  8302.234668  1294.800049        10.599976        -0.004077  129

In [ ]:


#targets y features
targets = [f"Close_t+{i}" for i in range(1, 8)]  # 7 targets, 7 modelos
features = [col for col in df.columns if col not in targets + ['Date']]  # usamos todas las features excepto targets y date

#columnas a escalar
cols_scaler = features  # todas las numericas

#crear mapper
# mapper aplica: primero imputa valores nulos con la mediana, luego escala con robust scaler
mapper = DataFrameMapper([
    (cols_scaler, [SimpleImputer(strategy='median'), RobustScaler()])
], input_df=True, df_out=True)

#definir modelos
models_dict = {
    "LinearRegression": LinearRegression(),
    "KNN": KNeighborsRegressor(n_neighbors=15),
    "DecisionTree": DecisionTreeRegressor(max_depth=None, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42)
}

#pipelines vacios por cada target
pipelines = {}
for target in targets:
    # aca solo definimos el pipeline base, luego le metemos cada modelo de la lista, 28 pipelines
    pipelines[target] = {}
    
    for model_name, model in models_dict.items():
        pipeline = Pipeline([
            ('mapper', mapper),
            ('model', model)
        ])
        pipelines[target][model_name] = pipeline


In [ ]:

# definimos tamaño de test final
test_size = 0.1
n_test = int(len(df) * test_size)

# separamos test final
train_val_df = df.iloc[:-n_test].reset_index(drop=True)
test_df = df.iloc[-n_test:].reset_index(drop=True)

n_splits = 7
tscv = TimeSeriesSplit(n_splits=n_splits)

# diccionarios para guardar índices por fold y por target
folds_idx = {target: [] for target in targets}

print(f"División en {n_splits} folds (train/val) por fechas:\n")
for target in targets:
    print(f" Target: {target}")
    X = train_val_df[features]
    y = train_val_df[[target]]

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
        # guardar indices
        folds_idx[target].append((train_idx, val_idx))

        # fechas para mostrar
        start_train = train_val_df.iloc[train_idx[0]]['Date']
        end_train   = train_val_df.iloc[train_idx[-1]]['Date']
        start_val   = train_val_df.iloc[val_idx[0]]['Date']
        end_val     = train_val_df.iloc[val_idx[-1]]['Date']

        print(f"Fold {fold+1}:")
        print(f"  Train: {start_train.date()} -> {end_train.date()} ({len(train_idx)} filas)")
        print(f"  Val:   {start_val.date()} -> {end_val.date()} ({len(val_idx)} filas)\n")

# mostrar tamaño test final
print(f"Test final: {test_df['Date'].min().date()} -> {test_df['Date'].max().date()} ({len(test_df)} filas)")


División en 7 folds (train/val) por fechas:

 Target: Close_t+1
Fold 1:
  Train: 2018-06-01 -> 2019-03-29 (302 filas)
  Val:   2019-03-30 -> 2020-01-25 (302 filas)

Fold 2:
  Train: 2018-06-01 -> 2020-01-25 (604 filas)
  Val:   2020-01-26 -> 2020-11-22 (302 filas)

Fold 3:
  Train: 2018-06-01 -> 2020-11-22 (906 filas)
  Val:   2020-11-23 -> 2021-09-20 (302 filas)

Fold 4:
  Train: 2018-06-01 -> 2021-09-20 (1208 filas)
  Val:   2021-09-21 -> 2022-07-19 (302 filas)

Fold 5:
  Train: 2018-06-01 -> 2022-07-19 (1510 filas)
  Val:   2022-07-20 -> 2023-05-17 (302 filas)

Fold 6:
  Train: 2018-06-01 -> 2023-05-17 (1812 filas)
  Val:   2023-05-18 -> 2024-03-14 (302 filas)

Fold 7:
  Train: 2018-06-01 -> 2024-03-14 (2114 filas)
  Val:   2024-03-15 -> 2025-01-10 (302 filas)

 Target: Close_t+2
Fold 1:
  Train: 2018-06-01 -> 2019-03-29 (302 filas)
  Val:   2019-03-30 -> 2020-01-25 (302 filas)

Fold 2:
  Train: 2018-06-01 -> 2020-01-25 (604 filas)
  Val:   2020-01-26 -> 2020-11-22 (302 filas)

Fold

In [ ]:

n_splits = 7
tscv = TimeSeriesSplit(n_splits=n_splits)

# diccionario para resultados
cv_results = {}  # {target: {model_name: [fold_results]}}

for target in targets:
    print(f"\n Target: {target}")
    X = df[features]
    y = df[[target]]

    cv_results[target] = {}

    for model_name, pipeline in pipelines[target].items():
        print(f"\n Modelo: {model_name}")
        fold_results = []

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

            # entrenar
            pipeline.fit(X_train, y_train)

            # predecir
            y_pred_train = pipeline.predict(X_train)
            y_pred_val   = pipeline.predict(X_val)

            # métricas
            mae_train = mean_absolute_error(y_train, y_pred_train)
            rmse_train = mean_squared_error(y_train, y_pred_train)**0.5

            mae_val = mean_absolute_error(y_val, y_pred_val)
            rmse_val = mean_squared_error(y_val, y_pred_val)**0.5

            fold_results.append({
                "fold": fold+1,
                "MAE_train": mae_train, "RMSE_train": rmse_train,
                "MAE_val": mae_val, "RMSE_val": rmse_val
            })

            print(f"Fold {fold+1}: Train MAE {mae_train:.2f}, RMSE {rmse_train:.2f} | "
                  f"Val MAE {mae_val:.2f}, RMSE {rmse_val:.2f}")

        cv_results[target][model_name] = fold_results



 Target: Close_t+1

 Modelo: LinearRegression
Fold 1: Train MAE 113.60, RMSE 167.01 | Val MAE 440.94, RMSE 549.13
Fold 2: Train MAE 188.53, RMSE 302.98 | Val MAE 761.50, RMSE 1435.54
Fold 3: Train MAE 369.70, RMSE 725.89 | Val MAE 1900.78, RMSE 2428.56
Fold 4: Train MAE 742.29, RMSE 1220.76 | Val MAE 2046.90, RMSE 2273.69
Fold 5: Train MAE 734.99, RMSE 1190.40 | Val MAE 473.50, RMSE 692.40
Fold 6: Train MAE 689.43, RMSE 1120.63 | Val MAE 1371.28, RMSE 1854.65
Fold 7: Train MAE 792.27, RMSE 1245.64 | Val MAE 1991.04, RMSE 2612.09

 Modelo: KNN
Fold 1: Train MAE 162.55, RMSE 250.14 | Val MAE 3561.47, RMSE 3854.79
Fold 2: Train MAE 271.93, RMSE 438.34 | Val MAE 8117.00, RMSE 14132.69
Fold 3: Train MAE 426.18, RMSE 863.01 | Val MAE 4462.39, RMSE 5594.87
Fold 4: Train MAE 1105.01, RMSE 1875.53 | Val MAE 8451.54, RMSE 9627.43
Fold 5: Train MAE 1087.31, RMSE 1798.52 | Val MAE 6847.53, RMSE 8190.65
Fold 6: Train MAE 1114.35, RMSE 1770.44 | Val MAE 19271.14, RMSE 20986.91
Fold 7: Train MAE 135

In [ ]:

# features y targets ya definidos

# diccionario para guardar pesos por fold/epoch
model_weights_by_horizon = {}

# Callback para guardar pesos
class OurCustomCallback(Callback):
    def __init__(self, horizon):
        super().__init__()
        self.horizon = horizon
        
    def on_epoch_end(self, epoch, logs=None):
        import copy
        if self.horizon not in model_weights_by_horizon:
            model_weights_by_horizon[self.horizon] = {}
        model_weights_by_horizon[self.horizon][epoch] = copy.deepcopy(self.model.get_weights())

# función para crear un modelo base
def create_mlp_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(1, activation='linear')  # salida para un solo target
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae','mse'])
    return model


: 

In [ ]:

# para guardar resultados
cv_results_nn = {}

# escalador
scaler = RobustScaler()

# early stopping para evitar sobreajuste
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

for i, target in enumerate(targets, start=1):
    print(f"\n Entrenando modelo NN para {target}")
    cv_results_nn[target] = []

    X = train_val_df[features].values
    y = train_val_df[[target]].values

    for fold, (train_idx, val_idx) in enumerate(folds_idx[target]):
        print(f"\n Fold {fold+1}/{len(folds_idx[target])}")

        # separar sets
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # escalar usando solo el train
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        # crear nuevo modelo
        model = create_mlp_model(X_train_scaled.shape[1])

        # callback personalizado (opcional)
        callback = OurCustomCallback(horizon=i)

        # entrenar
        history = model.fit(
            X_train_scaled, y_train,
            validation_data=(X_val_scaled, y_val),
            epochs=150,
            batch_size=32,
            verbose=0
        )

        # predicciones
        y_pred_train = model.predict(X_train_scaled)
        y_pred_val = model.predict(X_val_scaled)

        # métricas
        mae_train = mean_absolute_error(y_train, y_pred_train)
        rmse_train = mean_squared_error(y_train, y_pred_train) ** 0.5
        mape_train = mean_absolute_percentage_error(y_train, y_pred_train)

        mae_val = mean_absolute_error(y_val, y_pred_val)
        rmse_val = mean_squared_error(y_val, y_pred_val) ** 0.5
        mape_val = mean_absolute_percentage_error(y_val, y_pred_val)

        print(
            f"Fold {fold+1}: "
            f"Train -> MAE={mae_train:.4f}, RMSE={rmse_train:.4f} | "
            f"Val -> MAE={mae_val:.4f}, RMSE={rmse_val:.4f}"
        )

        # guardar resultados
        cv_results_nn[target].append({
            'fold': fold+1,
            'MAE_train': mae_train,
            'RMSE_train': rmse_train,
            'MAPE_train': mape_train,
            'MAE_val': mae_val,
            'RMSE_val': rmse_val,
            'MAPE_val': mape_val
        })

# promedio final por target
print("\n Resultados Promedio por Horizonte")
for target in targets:
    mae_train_mean = np.mean([r['MAE_train'] for r in cv_results_nn[target]])
    mae_val_mean = np.mean([r['MAE_val'] for r in cv_results_nn[target]])
    rmse_train_mean = np.mean([r['RMSE_train'] for r in cv_results_nn[target]])
    rmse_val_mean = np.mean([r['RMSE_val'] for r in cv_results_nn[target]])

    print(
        f"{target}: "
        f"Train -> MAE={mae_train_mean:.4f}, RMSE={rmse_train_mean:.4f} | "
        f"Val -> MAE={mae_val_mean:.4f}, RMSE={rmse_val_mean:.4f}"
    )



 Entrenando modelo NN para Close_t+1

 Fold 1/7
10/10 [==============================] - 0s 778us/step
Fold 1: Train -> MAE=218.9271, RMSE=288.0855 | Val -> MAE=1747.8639, RMSE=2295.4667

 Fold 2/7
10/10 [==============================] - 0s 889us/step
Fold 2: Train -> MAE=249.4762, RMSE=327.3694 | Val -> MAE=3999.0033, RMSE=4186.9969

 Fold 3/7
10/10 [==============================] - 0s 777us/step
Fold 3: Train -> MAE=217.1883, RMSE=316.5814 | Val -> MAE=4426.5974, RMSE=5437.7210

 Fold 4/7
10/10 [==============================] - 0s 835us/step
Fold 4: Train -> MAE=607.5762, RMSE=1085.8038 | Val -> MAE=1406.4640, RMSE=1776.1148

 Fold 5/7
10/10 [==============================] - 0s 778us/step
Fold 5: Train -> MAE=804.1718, RMSE=1275.5497 | Val -> MAE=753.9050, RMSE=969.6031

 Fold 6/7
10/10 [==============================] - 0s 889us/step
Fold 6: Train -> MAE=782.5331, RMSE=1176.4185 | Val -> MAE=1091.9444, RMSE=1541.5072

 Fold 7/7
10/10 [==============================] - 0s 889us/

In [ ]:
import os
import joblib  # para guardar el scaler

# directorio para guardar modelos
output_dir = "./models_final"
os.makedirs(output_dir, exist_ok=True)

# diccionario para guardar modelos y resultados
final_models = {}
final_metrics = {}

# iterar sobre los 7 horizontes
for i, target in enumerate(targets, start=1):
    print(f"\n Entrenando modelo final para {target}")
    
    # X e y completos (train + val)
    X = train_val_df[features].values
    y = train_val_df[[target]].values

    # escalar
    scaler_final = RobustScaler()
    X_scaled = scaler_final.fit_transform(X)
    
    # crear modelo
    model = create_mlp_model(X_scaled.shape[1])

    # entrenar
    history = model.fit(
        X_scaled, y,
        epochs=200,
        batch_size=32,
        verbose=0
    )

    # predicciones sobre todo el dataset
    y_pred = model.predict(X_scaled)

    # calcular metricas
    mae = mean_absolute_error(y, y_pred)
    rmse = mean_squared_error(y, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y, y_pred)

    print(f"{target} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

    # guardar modelo en h5
    model_path = os.path.join(output_dir, f"mlp_{target}.h5")
    save_model(model, model_path)
    print(f"modelo guardado en {model_path}")

    # guardar scaler asociado a este horizonte
    scaler_path = os.path.join(output_dir, f"scaler_{target}.pkl")
    joblib.dump(scaler_final, scaler_path)
    print(f"scaler guardado en {scaler_path}")

    # guardar en diccionario para uso inmediato
    final_models[target] = {
        "model": model,
        "scaler": scaler_final
    }

    # guardar metricas
    final_metrics[target] = {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

print("\n Todos los modelos finales, scalers y metricas guardados")
for target, met in final_metrics.items():
    print(f"{target}: MAE={met['MAE']:.4f}, RMSE={met['RMSE']:.4f}, MAPE={met['MAPE']:.4f}")



 Entrenando modelo final para Close_t+1
76/76 [==============================] - 0s 827us/step
Close_t+1 -> MAE: 807.0849, RMSE: 1264.8443, MAPE: 0.0336
modelo guardado en ./models_final\mlp_Close_t+1.h5
scaler guardado en ./models_final\scaler_Close_t+1.pkl

 Entrenando modelo final para Close_t+2
76/76 [==============================] - 0s 707us/step
Close_t+2 -> MAE: 1080.2570, RMSE: 1685.7350, MAPE: 0.0434
modelo guardado en ./models_final\mlp_Close_t+2.h5
scaler guardado en ./models_final\scaler_Close_t+2.pkl

 Entrenando modelo final para Close_t+3
76/76 [==============================] - 0s 733us/step
Close_t+3 -> MAE: 1314.9905, RMSE: 2009.0829, MAPE: 0.0503
modelo guardado en ./models_final\mlp_Close_t+3.h5
scaler guardado en ./models_final\scaler_Close_t+3.pkl

 Entrenando modelo final para Close_t+4
76/76 [==============================] - 0s 693us/step
Close_t+4 -> MAE: 1525.2772, RMSE: 2344.4086, MAPE: 0.0637
modelo guardado en ./models_final\mlp_Close_t+4.h5
scaler guard

In [ ]:


# diccionario para guardar métricas de test
test_metrics = {}

for target in targets:
    print(f"\n Evaluando modelo en test para {target}")

    # obtener modelo y scaler desde final_models
    model = final_models[target]["model"]
    scaler = final_models[target]["scaler"]

    # preparar X e y de test
    X_test = test_df[features].values
    y_test = test_df[[target]].values

    # escalar usando el scaler
    X_test_scaled = scaler.transform(X_test)

    # predecir
    y_pred_test = model.predict(X_test_scaled)

    # calcular métricas
    mae = mean_absolute_error(y_test, y_pred_test)
    rmse = mean_squared_error(y_test, y_pred_test) ** 0.5
    mape = mean_absolute_percentage_error(y_test, y_pred_test)

    print(f"{target} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, MAPE: {mape:.4f}")

    # guardar métricas
    test_metrics[target] = {
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape
    }

print("\n métricas finales sobre test")
for target, met in test_metrics.items():
    print(f"{target}: MAE={met['MAE']:.4f}, RMSE={met['RMSE']:.4f}, MAPE={met['MAPE']:.4f}")



 Evaluando modelo en test para Close_t+1
9/9 [==============================] - 0s 750us/step
Close_t+1 -> MAE: 1793.9860, RMSE: 2337.0098, MAPE: 0.0179

 Evaluando modelo en test para Close_t+2
9/9 [==============================] - 0s 875us/step
Close_t+2 -> MAE: 2306.1575, RMSE: 3061.1353, MAPE: 0.0228

 Evaluando modelo en test para Close_t+3
9/9 [==============================] - 0s 875us/step
Close_t+3 -> MAE: 2768.5951, RMSE: 3599.8844, MAPE: 0.0277

 Evaluando modelo en test para Close_t+4
9/9 [==============================] - 0s 1000us/step
Close_t+4 -> MAE: 2988.8505, RMSE: 3910.2352, MAPE: 0.0294

 Evaluando modelo en test para Close_t+5
9/9 [==============================] - 0s 875us/step
Close_t+5 -> MAE: 3366.2247, RMSE: 4326.5930, MAPE: 0.0331

 Evaluando modelo en test para Close_t+6
9/9 [==============================] - 0s 875us/step
Close_t+6 -> MAE: 3630.1895, RMSE: 4634.5240, MAPE: 0.0357

 Evaluando modelo en test para Close_t+7
9/9 [============================